KWANELE PRETTY MNGADI (mngadik58@gmail.com)

In [ ]:
import pandas as pd
from pandas import *
import numpy as np
from numpy import*

In [ ]:
# Importing the data
df = read_excel('/content/Dataset for Data Analytics.xlsx')

In [ ]:
df

In [ ]:
df.info()

**INITIAL AUDIT**

In [ ]:
print("columns:",len(df.columns))
print ('rows:',len(df))

Phase 1 : Strategic imputation

In [ ]:
print(df.isnull().sum())

In [ ]:
for col in df.columns:
  if df[col].isnull().sum()>0:
    missing_count = df[col].isnull().sum()
    if pd.api.types.is_numeric_dtype(df[col]):

      # for the numerical columns
      if abs(df[col].skew())>1:
        df[col]= df[col].fillna(df[col].median())
        print(f"{col}: Missing values filled using median")
      else:
        df[col] = df[col].fillna(df[col].mean())
        print(f"{col}:{missing_count} missing values filled using mean")
    else:
      df[col] = df[col].fillna(df[col].mode()[0])
      print(f"{col}:{missing_count}missing values filled using mode")






Phase 2: INTEGRITY AUDIT (remove duplicate)

In [ ]:
duplicated_before = df.duplicated().sum()
print("Duplicated rows before cleaning:",duplicated_before)

df = df.drop_duplicates()
audit_log =[]

audit_log.append({"Change ID": f"CR{len(audit_log)+1:03d}", "Description":"Removed duplicated rows","impact":f"Removed{duplicated_before} duplicate records","Status":"Resolved"})
print("Duplicate rows after cleaning:", df.duplicated().sum())


Phase 3: SPEAK ONE LANGUAGE

In [ ]:
# Clean text columns.

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
  df[col] =df[col].astype(str).str.strip()
  df[col] = df[col].str.replace(r"\s+", " ", regex=True)
  df[col] = df[col].str.title()
audit_log.append({"Change ID": f"CR{len(audit_log)+1:03d}","Description": "Standardized text columns using trim, spacing, and title case","Impact": f"Cleaned {len(text_cols)} text columns","Status": "Resolved"})


Phase 3B: DATE FORMAT CLEANING

In [ ]:
possible_date_cols = [col for col in df.columns if "date" in col.lower() or "time" in col.lower()]
invalid_dates = {}
for col in possible_date_cols:
  before_invalid = df[col].isnull().sum()
  df[col] = pd.to_datetime(df[col], errors="coerce")
  after_invalid = df[col].isnull().sum()
  invalid_dates[col] = after_invalid
  df[col] = df[col].dt.strftime("%d-%m-%Y")


audit_log.append({"Change ID": f"CR{len(audit_log)+1:03d}","Description": f"Converted {col} to YYYY-MM-DD date format","Impact": f"{after_invalid} invalid dates detected after conversion","Status": "Resolved"})
print("Date columns checked:", possible_date_cols)

Phase 3C : NUMERIC FORMAT CLEANING

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
for col in numeric_cols:
  df[col] = pd.to_numeric(df[col], errors="coerce")
if df[col].dtype == "float64":
  df[col] = df[col].round(2)
audit_log.append({"Change ID": f"CR{len(audit_log)+1:03d}","Description": "Standardized numeric columns to 2 decimal precision","Impact": f"Checked {len(numeric_cols)} numeric columns","Status": "Resolved"})

FINAL VERIFICATION

In [ ]:
final_missing = df.isnull().sum().sum()
final_duplicates = df.duplicated().sum()
print("Missing Values Remaining:", final_missing)
print("Duplicate Rows Remaining:", final_duplicates)
if len(possible_date_cols) > 0:
  print("Date Columns Standardized:", possible_date_cols)
else:
  print("No date columns detected")
print("Final shape:", df.shape)

CREATE AUDIT LOG

In [ ]:
audit_df =pd.DataFrame(audit_log)
audit_df

EXPORT CLEANED DATA AND AUDIT LOG

In [ ]:
df.to_excel("Cleaned_dataset.xlsx",index=False)